In [1]:
import os

In [2]:
%pwd

'/mnt/d/dl_project/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/mnt/d/dl_project'

In [5]:

import mlflow


/mnt/d/dl_project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from dotenv import load_dotenv
dotenv_path = "/mnt/d/dl_project/.env"
load_dotenv(dotenv_path)

True

In [7]:
token = os.getenv("DAGSHUB_TOKEN")



In [9]:

import dagshub
dagshub.auth.add_app_token(token)
dagshub.init(repo_owner='pavani96-ai', repo_name='kidney_disease_classification', mlflow=True)


The added token already exists in the token cache, skipping


Accessing as pavani96-ai

Initialized MLflow to track repo "pavani96-ai/kidney_disease_classification"

Repository pavani96-ai/kidney_disease_classification initialized!

In [10]:
os.environ["MLFLOW_TRACKING_URI"] = os.getenv("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD")



In [11]:
import tensorflow as tf


I0000 00:00:1782370532.435452    2271 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782370534.451707    2271 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782370565.356507    2271 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [13]:
model = tf.keras.models.load_model("artifacts/training/model.keras")

I0000 00:00:1782370718.196034    2271 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3539 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [19]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class  EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int
    valid_score: Path

In [20]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [29]:
class Configurationmanager:
    def __init__(self, 
                 config_filepath= CONFIG_FILE_PATH,
                 params_filepath= PARAMS_FILE_PATH,
                 schema_filepath= SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    def get_evaluation_config(self) -> EvaluationConfig:
        config = self.config.model_evaluation
        params =self.params
        
        create_directories([config.root_dir])
        eval_config = EvaluationConfig(
            path_of_model = Path(config.path_of_model),
            training_data = Path(config.training_data),
            mlflow_uri = os.getenv("MLFLOW_URI"),
            all_params = params,
            params_image_size = params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
            valid_score = Path(config.valid_score)
        )
        return eval_config


In [30]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [39]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
    
    def valid_generator(self):
        datagenerator_kwargs = dict(
            validation_split = 0.30
        )
        
        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_generator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_generator.flow_from_directory(
            directory = self.config.training_data,
            subset = "validation",
            shuffle=False,
            **dataflow_kwargs)
        
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self.valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0],
                   "accuracy": self.score[1],
                   "f1_score": self.score[2]}
        save_json(path=Path(self.config.valid_score), data=scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0],
                   "accuracy": self.score[1],
                   "f1_score": self.score[2]}
            )
            if tracking_url_type_store != "file":
                #Register the model
                mlflow.keras.log_model(self.model, "model", registered_model_name="MobileNetV2")

            else:
                mlflow.keras.log_model(self.model, "model")
                

    
    
    



In [40]:
try: 
    config = Configurationmanager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
    raise e


[2026-06-25 08:19:11,804: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-25 08:19:11,811: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-25 08:19:11,816: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-25 08:19:11,821: INFO: common: created directory at: artifacts]
[2026-06-25 08:19:11,827: INFO: common: created directory at: artifacts/model_evaluation]
Found 3732 images belonging to 4 classes.


I0000 00:00:1782375554.682502    2271 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1782375556.882234    4325 service.cc:153] XLA service 0x7f2004032010 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782375556.882342    4325 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 12.8.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.2)
I0000 00:00:1782375557.133126    4325 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782375557.995187    4325 cuda_dnn.cc:461] Loaded cuDNN version 92302
E0000 00:00:1782375565.732941    4325 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1782375592.399467    4325 device_compiler.h:208] Compiled cluster 

116/117 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step - accuracy: 0.9625 - f1_score: 0.5279 - loss: 0.1296

E0000 00:00:1782375643.635858    4325 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


117/117 ━━━━━━━━━━━━━━━━━━━━ 108s 603ms/step - accuracy: 0.9780 - f1_score: 0.9704 - loss: 0.1007
[2026-06-25 08:21:02,760: INFO: common: json file saved at: artifacts/model_evaluation/scores.json]


2026/06/25 08:21:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/25 08:21:11 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Successfully registered model 'MobileNetV2'.
2026/06/25 08:23:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: MobileNetV2, version 1
Created version '1' of model 'MobileNetV2'.


🏃 View run defiant-newt-339 at: https://dagshub.com/pavani96-ai/kidney_disease_classification.mlflow/#/experiments/0/runs/b5a5777437f64d14b565b19ae62249cd
🧪 View experiment at: https://dagshub.com/pavani96-ai/kidney_disease_classification.mlflow/#/experiments/0
